FEATURE ENGINEERING

In [1]:
import polars as pl

auth = pl.scan_parquet("filtered_auth.parquet")

auth_users = (
    auth
    .filter(pl.col("src_user").str.starts_with("U"))
    .rename({"src_user": "user"})
)
user_features = (
    auth_users
    .group_by("user")
    .agg([
        pl.len().alias("total_logins"),
        pl.col("status").filter(pl.col("status") == "Failure").count().alias("failed_logins"),
        pl.col("dst_computer").n_unique().alias("unique_destinations")
    ])
)


# Redteam data extraction
red = pl.scan_csv(
    "data_windows/redteam.txt.gz",
    has_header=False,
    new_columns=["time", "user", "src_computer", "dst_computer"]
)

comp_users = (
    red
    .group_by("user")
    .agg([
        pl.len().alias("compromise_events")
    ])
)



In [2]:
user_stats = (
    user_features
    .join(comp_users, on="user", how="left")
    .with_columns(
        pl.col("compromise_events").fill_null(0),
    )
)

user_stats_df = user_stats.collect(streaming=True)

print(user_stats_df.shape)
print(user_stats_df.sort("compromise_events", descending=True).head())

user_features = (
    auth_users
    .group_by("user")
    .agg([
        pl.len().alias("total_logins"),
        pl.col("status").filter(pl.col("status") == "Failure").count().alias("failed_logins"),
        pl.col("dst_computer").n_unique().alias("unique_destinations"),
        pl.col("src_computer").n_unique().alias("unique_sources"),
    ])
)
user_stats = user_stats.with_columns(
    (pl.col("failed_logins") / pl.col("total_logins")).alias("failure_ratio")
)


C:\Users\kruti\AppData\Local\Temp\ipykernel_23420\2319990529.py:9: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  user_stats_df = user_stats.collect(streaming=True)


(25951, 5)
shape: (5, 5)
┌────────────┬──────────────┬───────────────┬─────────────────────┬───────────────────┐
│ user       ┆ total_logins ┆ failed_logins ┆ unique_destinations ┆ compromise_events │
│ ---        ┆ ---          ┆ ---           ┆ ---                 ┆ ---               │
│ str        ┆ u32          ┆ u32           ┆ u32                 ┆ u32               │
╞════════════╪══════════════╪═══════════════╪═════════════════════╪═══════════════════╡
│ U66@DOM1   ┆ 4977072      ┆ 0             ┆ 238                 ┆ 118               │
│ U3005@DOM1 ┆ 18068        ┆ 0             ┆ 66                  ┆ 36                │
│ U737@DOM1  ┆ 27203        ┆ 0             ┆ 129                 ┆ 32                │
│ U293@DOM1  ┆ 100265       ┆ 0             ┆ 78                  ┆ 31                │
│ U1653@DOM1 ┆ 88210        ┆ 0             ┆ 3362                ┆ 31                │
└────────────┴──────────────┴───────────────┴─────────────────────┴───────────────────┘


In [3]:
import pandas as pd

df = user_stats_df.to_pandas()

# Features only
features = df[[
    "total_logins",
    "failed_logins",
    "unique_destinations"
]]

labels = (df["compromise_events"] > 0).astype(int)


ISOLATION FOREST MODEL

In [4]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    n_estimators=200,
    contamination=0.01,   # assume ~1% anomalies
    random_state=42
)

iso.fit(features)

df["anomaly_score"] = iso.decision_function(features)
df["anomaly_flag"] = iso.predict(features)


In [5]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(labels, -df["anomaly_score"])
print("ROC-AUC:", auc)


ROC-AUC: 0.882947656475532


In [6]:
df_sorted = df.sort_values("anomaly_score")

print(df_sorted.head(20)[[
    "user",
    "anomaly_score",
    "compromise_events"
]])


             user  anomaly_score  compromise_events
18747    U66@DOM1      -0.133633                118
5513     U78@DOM1      -0.125020                  2
21696     U8@DOM1      -0.118192                  0
1042     U24@DOM1      -0.112826                  5
10704    U94@DOM1      -0.106376                  0
25506    U13@DOM1      -0.104980                  2
22911   U726@DOM1      -0.102474                  0
521     U194@DOM1      -0.100808                  0
9199     U19@DOM1      -0.100498                  0
8656     U48@DOM1      -0.100498                  0
20143  U1653@DOM1      -0.099976                 31
16326    U34@DOM1      -0.098314                  0
17364   U346@DOM1      -0.097802                  0
23085  U7056@DOM1      -0.097209                  0
2856     U54@DOM1      -0.096673                  0
12917   U200@DOM1      -0.096656                  0
1089     U87@DOM1      -0.095002                  0
23695   U113@DOM1      -0.094767                  0
20506    U14

In [7]:
# Top 50 most anomalous users
top_50 = df.sort_values("anomaly_score").head(50)

print("Compromised in Top 50:", top_50["compromise_events"].gt(0).sum())
# How many compromised total?
print("Total compromised users:", labels.sum())


Compromised in Top 50: 7
Total compromised users: 104


In [8]:
import numpy as np

df_eval = df.copy()

df_eval["is_compromised"] = (df_eval["compromise_events"] > 0).astype(int)

def precision_at_k(df, k):
    top_k = df.sort_values("anomaly_score").head(k)
    return top_k["is_compromised"].sum() / k

for k in [10, 20, 50, 100, 200]:
    print(f"Precision@{k}: {precision_at_k(df_eval, k):.4f}")


Precision@10: 0.4000
Precision@20: 0.2500
Precision@50: 0.1400
Precision@100: 0.1100
Precision@200: 0.1200


In [9]:
df_behavior = df.copy()

# Convert anomaly score to percentile rank
df_behavior["behavioral_percentile"] = (
    df_behavior["anomaly_score"].rank(pct=True)
)

# Invert (lower anomaly_score = higher risk)
df_behavior["behavioral_percentile"] = 1 - df_behavior["behavioral_percentile"]

# Scale to 0–100
df_behavior["risk_score"] = df_behavior["behavioral_percentile"] * 100


In [10]:
print(df_behavior[["user", "risk_score"]].head())
print(df_behavior["risk_score"].describe())


          user  risk_score
0       U936@?   45.063774
1   U9635@DOM9   11.036184
2   U7603@DOM1   69.180378
3  U11796@DOM9   19.565720
4   U5766@DOM1   73.141690
count    25951.000000
mean        49.998073
std         28.866052
min          0.186891
25%         25.274556
50%         49.938345
75%         74.997110
max         99.996147
Name: risk_score, dtype: float64


TEMPORAL MODEL LAYER

In [11]:
import polars as pl

auth = pl.scan_parquet("filtered_auth.parquet")

auth_users = (
    auth
    .filter(pl.col("src_user").str.starts_with("U"))
    .rename({"src_user": "user"})
)

WINDOW_SIZE = 10000 

auth_windowed = auth_users.with_columns(
    (pl.col("time") // WINDOW_SIZE).alias("window_id")
)


In [12]:
window_features = (
    auth_windowed
    .group_by(["user", "window_id"])
    .agg([
        pl.len().alias("login_count"),
        pl.col("dst_computer").n_unique().alias("unique_destinations"),
        pl.col("status").filter(pl.col("status") == "Failure").count().alias("failure_count")
    ])
)
window_df = window_features.collect(streaming=True)

print(window_df.shape)
window_df.head()





C:\Users\kruti\AppData\Local\Temp\ipykernel_23420\2729512689.py:10: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  window_df = window_features.collect(streaming=True)


(1485841, 5)


user,window_id,login_count,unique_destinations,failure_count
str,i64,u32,u32,u32
"""U8960@DOM1""",207,1,1,0
"""U25@DOM1""",57,98,6,0
"""U9962@DOM1""",113,44,7,0
"""U1042@DOM1""",17,52,5,0
"""U4225@DOM1""",96,74,7,0


In [13]:
import numpy as np
from sklearn.ensemble import IsolationForest

df_w = window_df.to_pandas()

# Add ratio feature
df_w["dest_ratio"] = df_w["unique_destinations"] / df_w["login_count"]
df_w["failure_ratio"] = df_w["failure_count"] / df_w["login_count"]

df_w = df_w.fillna(0)

features = df_w[[
    "login_count",
    "unique_destinations",
    "failure_count",
    "dest_ratio",
    "failure_ratio"
]]

iso = IsolationForest(
    n_estimators=300,
    contamination=0.01,
    random_state=42
)

iso.fit(features)

df_w["anomaly_score"] = iso.decision_function(features)
df_w["anomaly_flag"] = iso.predict(features)
user_window_scores = (
    df_w.groupby("user")["anomaly_score"]
    .min()
    .reset_index()
)

user_window_scores = user_window_scores.rename(
    columns={"anomaly_score": "temporal_score"}
)

In [14]:
df_users = user_stats_df.to_pandas()

df_users["is_compromised"] = (df_users["compromise_events"] > 0).astype(int)

df_eval = df_users.merge(user_window_scores, on="user", how="left")

from sklearn.metrics import roc_auc_score

auc_temporal = roc_auc_score(
    df_eval["is_compromised"],
    -df_eval["temporal_score"]
)

print("Temporal ROC-AUC:", auc_temporal)
def precision_at_k(df, k):
    top_k = df.sort_values("temporal_score").head(k)
    return top_k["is_compromised"].sum() / k

for k in [10, 20, 50, 100]:
    print(f"Temporal Precision@{k}: {precision_at_k(df_eval, k):.4f}")


Temporal ROC-AUC: 0.7489126100038392
Temporal Precision@10: 0.3000
Temporal Precision@20: 0.2500
Temporal Precision@50: 0.2000
Temporal Precision@100: 0.1100


IMPROVING THE TEMPORAL LAYER 

In [15]:
df_w = df_w.sort_values(["user", "window_id"])

# Previous window stats per user
df_w["prev_login"] = df_w.groupby("user")["login_count"].shift(1)
df_w["prev_dest"] = df_w.groupby("user")["unique_destinations"].shift(1)

# Delta features
df_w["login_delta"] = df_w["login_count"] - df_w["prev_login"]
df_w["dest_delta"] = df_w["unique_destinations"] - df_w["prev_dest"]

df_w = df_w.fillna(0)

df_w["rolling_login_mean"] = (
    df_w.groupby("user")["login_count"]
    .rolling(window=3, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

df_w["login_spike_ratio"] = (
    df_w["login_count"] /
    (df_w["rolling_login_mean"] + 1e-6)
)



In [16]:
temporal_features = df_w[[
    "login_count",
    "unique_destinations",
    "failure_count",
    "dest_ratio",
    "failure_ratio",
    "login_delta",
    "dest_delta",
    "login_spike_ratio"
]]
iso = IsolationForest(
    n_estimators=300,
    contamination=0.01,
    random_state=42
)

iso.fit(temporal_features)

df_w["anomaly_score"] = iso.decision_function(temporal_features)
user_window_scores = (
    df_w.groupby("user")["anomaly_score"]
    .min()
    .reset_index()
)

user_window_scores.rename(
    columns={"anomaly_score": "temporal_score"},
    inplace=True
)


In [17]:
# User-level ground truth
df_users = user_stats_df.to_pandas()
df_users["is_compromised"] = (df_users["compromise_events"] > 0).astype(int)

# Merge improved temporal scores
df_eval = df_users.merge(user_window_scores, on="user", how="left")

# Fill missing users with 0 anomaly
df_eval["temporal_score"] = df_eval["temporal_score"].fillna(0)
from sklearn.metrics import roc_auc_score

auc_temporal = roc_auc_score(
    df_eval["is_compromised"],
    -df_eval["temporal_score"]
)

print("Improved Temporal ROC-AUC:", auc_temporal)
def precision_at_k(df, k):
    top_k = df.sort_values("temporal_score").head(k)
    return top_k["is_compromised"].sum() / k

for k in [10, 20, 50, 100]:
    print(f"Improved Temporal Precision@{k}: {precision_at_k(df_eval, k):.4f}")
total_compromised = df_eval["is_compromised"].sum()

def recall_at_k(df, k):
    top_k = df.sort_values("temporal_score").head(k)
    return top_k["is_compromised"].sum() / total_compromised

for k in [10, 20, 50, 100]:
    print(f"Improved Temporal Recall@{k}: {recall_at_k(df_eval, k):.4f}")


Improved Temporal ROC-AUC: 0.8115513703420423
Improved Temporal Precision@10: 0.3000
Improved Temporal Precision@20: 0.2500
Improved Temporal Precision@50: 0.2200
Improved Temporal Precision@100: 0.1800
Improved Temporal Recall@10: 0.0288
Improved Temporal Recall@20: 0.0481
Improved Temporal Recall@50: 0.1058
Improved Temporal Recall@100: 0.1731


Event-Level Drilldown

In [18]:
auth_windowed = auth_windowed.with_columns(
    pl.col("user")
    .str.split("@")
    .list.get(0)
    .alias("user_clean")
)


UPDATED

In [19]:
df_behavior = df_behavior.copy()

df_behavior["user_clean"] = df_behavior["user"].str.split("@").str[0]
most_suspicious_windows = (
    df_w.sort_values("anomaly_score")
    .groupby("user", as_index=False)
    .first()
)

most_suspicious_windows = most_suspicious_windows[[
    "user",
    "window_id",
    "login_count",
    "unique_destinations",
    "login_delta",
    "dest_delta",
    "login_spike_ratio",
    "anomaly_score"
]]

most_suspicious_windows["user_clean"] = (
    most_suspicious_windows["user"].str.split("@").str[0]
)
import polars as pl
most_suspicious_pl = pl.from_pandas(
    most_suspicious_windows[["user_clean", "window_id"]]
).lazy()

drilldown_lazy = (
    auth_windowed
    .join(
        most_suspicious_pl,
        on=["user_clean", "window_id"],
        how="inner"
    )
    .group_by(["user_clean", "window_id"])
    .agg([
        pl.col("dst_computer").unique().alias("machines"),
        pl.count().alias("events_in_window")
    ])
)
window_machine_map = drilldown_lazy.collect(streaming=True)
window_machine_map_pd = window_machine_map.to_pandas()

C:\Users\kruti\AppData\Local\Temp\ipykernel_23420\2126479903.py:39: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("events_in_window")
C:\Users\kruti\AppData\Local\Temp\ipykernel_23420\2126479903.py:42: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  window_machine_map = drilldown_lazy.collect(streaming=True)


In [20]:
df_context = most_suspicious_windows.merge(
    window_machine_map_pd,
    on=["user_clean", "window_id"],
    how="left"
)
df_context = df_context.merge(
    df_behavior[["user_clean", "risk_score"]],
    on="user_clean",
    how="left"
)
user_window_scores_pd = user_window_scores.copy()
user_window_scores_pd["user_clean"] = (
    user_window_scores_pd["user"].str.split("@").str[0]
)

df_context = df_context.merge(
    user_window_scores_pd[["user_clean", "temporal_score"]],
    on="user_clean",
    how="left"
)

In [21]:
print("Rows:", df_context.shape[0])
print("Unique users:", df_context["user_clean"].nunique())
print(df_context[["user_clean","machines","risk_score","temporal_score"]].head())

Rows: 73396039
Unique users: 11720
  user_clean                                           machines  risk_score  \
0     U10000  [C10931, C529, C586, TGT, C1065, C612, C457, C...   56.839813   
1     U10000  [C10931, C529, C586, TGT, C1065, C612, C457, C...   56.839813   
2     U10000  [C10931, C529, C586, TGT, C1065, C612, C457, C...   44.169782   
3     U10000  [C10931, C529, C586, TGT, C1065, C612, C457, C...   44.169782   
4     U10000                               [C1115, C457, C1114]   56.839813   

   temporal_score  
0        0.121822  
1        0.111519  
2        0.121822  
3        0.111519  
4        0.121822  


In [22]:
top_alert = df_context.sort_values("temporal_score").iloc[0]

print("User:", top_alert["user"])
print("User clean:", top_alert["user_clean"])
print("Window:", top_alert["window_id"])
print("Machines field raw:", top_alert["machines"])
print("Type:", type(top_alert["machines"]))

User: U6836@C15244
User clean: U6836
Window: 125
Machines field raw: ['C12564' 'C20714' 'C4267' ... 'C5030' 'C5369' 'C2260']
Type: <class 'numpy.ndarray'>


In [23]:
window_machine_map_pd[
    (window_machine_map_pd["user_clean"] == top_alert["user_clean"]) &
    (window_machine_map_pd["window_id"] == top_alert["window_id"])
]

,user_clean,window_id,machines,events_in_window
10069,U6836,125,"[C12564, C20714, C4267, C9810, C891, C14166, C...",6787


In [24]:
import numpy as np

def generate_incident_report_with_context(row):
    machines = row.get("machines")

    if isinstance(machines, (list, np.ndarray)):
        machines_preview = list(machines)[:10]
    else:
        machines_preview = []

    risk = row.get("risk_score")
    risk_str = f"{risk:.2f}" if isinstance(risk, (int, float)) else "N/A"

    report = f"""
    Incident Report
    ----------------
    User: {row['user']}
    Risk Score: {risk_str}

    Most Suspicious Window: {row['window_id']}

    Window Indicators:
    - Login Count: {row['login_count']}
    - Unique Machines: {row['unique_destinations']}
    - Login Delta: {row['login_delta']}
    - Machine Delta: {row['dest_delta']}
    - Spike Ratio: {row['login_spike_ratio']:.2f}
    - Events in Window: {row.get('events_in_window', 'N/A')}

    Machines Accessed (Top 10):
    {machines_preview}

    Assessment:
    Sudden behavioral spike detected relative to historical baseline.
    Pattern consistent with potential lateral movement.

    Recommended Actions:
    - Temporarily disable account
    - Reset credentials
    - Audit accessed hosts
    """
    return report

In [25]:
top_alert = df_context.sort_values("temporal_score").iloc[0]
print(generate_incident_report_with_context(top_alert))


    Incident Report
    ----------------
    User: U6836@C15244
    Risk Score: 99.38

    Most Suspicious Window: 125

    Window Indicators:
    - Login Count: 6787
    - Unique Machines: 6735
    - Login Delta: 4764.0
    - Machine Delta: 4731.0
    - Spike Ratio: 2.31
    - Events in Window: 6787

    Machines Accessed (Top 10):
    ['C12564', 'C20714', 'C4267', 'C9810', 'C891', 'C14166', 'C9514', 'C10860', 'C16633', 'C2035']

    Assessment:
    Sudden behavioral spike detected relative to historical baseline.
    Pattern consistent with potential lateral movement.

    Recommended Actions:
    - Temporarily disable account
    - Reset credentials
    - Audit accessed hosts
    


SEVERITY DESIGN

In [26]:
# Convert temporal_score to percentile
df_context["temporal_percentile"] = (
    df_context["temporal_score"].rank(pct=True)
)

# Invert (lower temporal_score = higher anomaly)
df_context["temporal_percentile"] = 1 - df_context["temporal_percentile"]

df_context["temporal_risk"] = df_context["temporal_percentile"] * 100

In [27]:
df_context["final_risk"] = (
    0.6 * df_context["risk_score"] +
    0.4 * df_context["temporal_risk"]
)

In [28]:
import numpy as np

conditions = [
    (df_context["final_risk"] >= 95) & (df_context["unique_destinations"] > 100),
    (df_context["final_risk"] >= 85),
    (df_context["final_risk"] >= 70)
]

choices = ["CRITICAL", "HIGH", "MEDIUM"]

df_context["severity"] = np.select(conditions, choices, default="LOW")

In [29]:
import numpy as np

def generate_incident_report_with_context(row):
    machines = row.get("machines")
    severity = row.get("severity", "UNKNOWN")
    if isinstance(machines, (list, np.ndarray)):
        machines_preview = list(machines)[:10]
    else:
        machines_preview = []

    risk = row.get("risk_score")
    risk_str = f"{risk:.2f}" if isinstance(risk, (int, float)) else "N/A"

    report = f"""
    Incident Report
    ----------------
    User: {row['user']}
    Risk Score: {risk_str}

    Most Suspicious Window: {row['window_id']}

    Window Indicators:
    - Login Count: {row['login_count']}
    - Unique Machines: {row['unique_destinations']}
    - Login Delta: {row['login_delta']}
    - Machine Delta: {row['dest_delta']}
    - Spike Ratio: {row['login_spike_ratio']:.2f}
    - Events in Window: {row.get('events_in_window', 'N/A')}
    -Severity: {severity}
    Machines Accessed (Top 10):
    {machines_preview}

    Assessment:
    Sudden behavioral spike detected relative to historical baseline.
    Pattern consistent with potential lateral movement.

    Recommended Actions:
    - Temporarily disable account
    - Reset credentials
    - Audit accessed hosts
    """
    return report


In [30]:
top_alert = df_context.sort_values("final_risk", ascending=False).iloc[0]
print(generate_incident_report_with_context(top_alert))


    Incident Report
    ----------------
    User: U66@?
    Risk Score: 100.00

    Most Suspicious Window: 192

    Window Indicators:
    - Login Count: 1
    - Unique Machines: 1
    - Login Delta: -1.0
    - Machine Delta: 0.0
    - Spike Ratio: 0.33
    - Events in Window: 20746
    -Severity: HIGH
    Machines Accessed (Top 10):
    ['C2084', 'C245', 'C779', 'C2368', 'C815', 'C2186', 'C1581', 'C1152', 'C1703', 'C2149']

    Assessment:
    Sudden behavioral spike detected relative to historical baseline.
    Pattern consistent with potential lateral movement.

    Recommended Actions:
    - Temporarily disable account
    - Reset credentials
    - Audit accessed hosts
    
